🟢 Débutant | ⏱ 2 h 30 | 🔑 sqlite3, CREATE TABLE, INSERT, `?`, SELECT, UPDATE, DELETE, read_sql

# Module 12 — Exercices : ranger les données en base

Cours associé : [01-cours.ipynb](01-cours.ipynb).

## Comment travailler

1. Lis l'**objectif** et les **étapes**.
2. Remplace chaque `...` par ton code ; garde les noms de variables déjà écrits, la vérification en a besoin.
3. Exécute ta cellule (`Shift + Entrée`) et compare au **résultat attendu**.
4. Exécute la cellule **Vérification** : `✅ OK` = réussi ; sinon lis le message, il te dit quoi corriger.
   (Tu n'as pas besoin de comprendre le code des cellules Vérification : elles ouvrent ta base avec une
   **nouvelle** connexion, comme un collègue. Elles ne voient donc que ce que tu as validé avec `commit`.)
5. Bloqué plus de 5 minutes ? Ouvre l'indice 💡, puis relis la section du cours indiquée.

**Lire un échec.** Si la vérification échoue, la dernière ligne commence par `AssertionError:`. Ça veut dire « la vérification a échoué » (ce n'est pas Python qui est cassé). Le message **en français** après les deux-points te dit quoi corriger ; les lignes en anglais au-dessus, tu peux les ignorer. Les caractères spéciaux (`°`, `—`, `«`, `î`…) : copie-colle-les depuis l'énoncé.

Chaque exercice crée **sa propre base** dans `sorties/` (par exemple `sorties/ex_1_3.db`) et la recrée à neuf à
chaque exécution : tu peux relancer une cellule autant de fois que tu veux.

### ▶️ Cellule de préparation : exécute-la avant tout

Beaucoup d'exercices partent d'une base déjà remplie des 180 relevés. Pour ne pas répéter sept lignes à chaque fois,
on les range ici, une fois pour toutes, dans une fonction `preparer_base(chemin)` (module 07) : elle crée la base
`chemin` à neuf, la remplit avec `meteo.csv`, valide, et **renvoie la connexion**. C'est la fonction `creer_base` du
cours, section 4.

Si un exercice affiche `NameError: name 'preparer_base' is not defined`, c'est que cette cellule n'a pas été exécutée
(par exemple après un redémarrage du kernel) : exécute-la, puis relance l'exercice.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

Path("sorties").mkdir(exist_ok=True)


def preparer_base(chemin):
    connexion = sqlite3.connect(chemin)
    connexion.execute("DROP TABLE IF EXISTS releves")
    connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
    for f in pd.read_csv("../donnees/meteo.csv").to_dict("records"):
        connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", (f["date"], f["ville"], f["temp_min"], f["temp_max"], f["pluie_mm"], f["vent_kmh"]))
    connexion.commit()
    return connexion


print("Dossier sorties et fonction preparer_base prêts")

## Niveau 1 — Imiter

Le même geste que dans le cours, dans un cas presque identique.

### 1.1 Ouvrir une base

**Objectif** : Créer un fichier de base SQLite avec `connect`, puis fermer la connexion.

**À faire** :
1. Ouvre `sorties/ex_1_1.db` avec `sqlite3.connect` dans `connexion`.
2. Ferme-la avec `connexion.close()`.

**Résultat attendu** :
```
True
```

<details><summary>💡 Indice</summary>

`sqlite3.connect("sorties/ex_1_1.db")` puis `connexion.close()`.

📖 Cours, section 2.

</details>

In [ ]:
import sqlite3
from pathlib import Path

Path("sorties").mkdir(exist_ok=True)
connexion = sqlite3.connect(...)
...
print(Path("sorties/ex_1_1.db").exists())

In [ ]:
# Vérification
from pathlib import Path
assert Path("sorties/ex_1_1.db").exists(), "Le fichier sorties/ex_1_1.db n'existe pas : sqlite3.connect(\"sorties/ex_1_1.db\")."
print("✅ OK")

### 1.2 Créer une table de villes

**Objectif** : Créer une table `villes` à deux colonnes de texte.

**À faire** :
1. Supprime la table si elle existe : `DROP TABLE IF EXISTS villes`.
2. Crée-la : `CREATE TABLE villes (nom TEXT, departement TEXT)`.
3. Valide avec `commit`, puis ferme.

**Résultat attendu** :
```
Table villes créée
```

<details><summary>💡 Indice</summary>

La requête est une chaîne : `"CREATE TABLE villes (nom TEXT, departement TEXT)"`.

📖 Cours, section 3.

</details>

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/ex_1_2.db")
connexion.execute("DROP TABLE IF EXISTS villes")
connexion.execute(...)
connexion.commit()
connexion.close()
print("Table villes créée")

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_1_2.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "villes" in tables, "La table villes n'existe pas dans sorties/ex_1_2.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
colonnes = [c[1] for c in v.execute("PRAGMA table_info(villes)").fetchall()]
v.close()
assert colonnes == ["nom", "departement"], f"Les colonnes doivent être nom et departement, tu as {colonnes}."
print("✅ OK")

### 1.3 Ajouter une ville

**Objectif** : Ajouter une ligne avec une requête paramétrée.

**À faire** :
1. Ajoute la ville `"Albi"`, département `"Tarn"` : `INSERT INTO villes VALUES (?, ?)` et le tuple `("Albi", "Tarn")`.
2. N'oublie pas `commit`.

**Résultat attendu** :
```
[('Albi', 'Tarn')]
```

<details><summary>💡 Indice</summary>

Deuxième argument : le tuple `("Albi", "Tarn")`. Puis `connexion.commit()`.

📖 Cours, section 4.

</details>

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/ex_1_3.db")
connexion.execute("DROP TABLE IF EXISTS villes")
connexion.execute("CREATE TABLE villes (nom TEXT, departement TEXT)")
connexion.execute("INSERT INTO villes VALUES (?, ?)", ...)
...
print(connexion.execute("SELECT * FROM villes").fetchall())
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_1_3.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "villes" in tables, "La table villes n'existe pas dans sorties/ex_1_3.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
lignes = v.execute("SELECT * FROM villes").fetchall()
v.close()
assert lignes != [], "La table est vide pour une nouvelle connexion : as-tu fait connexion.commit() ?"
assert lignes == [("Albi", "Tarn")], f"La table doit contenir [('Albi', 'Tarn')], elle contient {lignes}."
print("✅ OK")

### 1.4 Ajouter plusieurs villes avec une boucle

**Objectif** : Faire un `INSERT` par élément d'une liste de tuples.

**À faire** :
1. Pour chaque tuple `ville` de `villes`, exécute l'`INSERT` avec `ville` comme valeurs.
2. Un seul `commit`, après la boucle.

**Résultat attendu** :
```
3
```

<details><summary>💡 Indice</summary>

`ville` est déjà un tuple de 2 valeurs : donne-le directement, `connexion.execute("INSERT ...", ville)`.

📖 Cours, section 4.

</details>

In [ ]:
import sqlite3

villes = [("Toulouse", "Haute-Garonne"), ("Nîmes", "Gard"), ("Perpignan", "Pyrénées-Orientales")]

connexion = sqlite3.connect("sorties/ex_1_4.db")
connexion.execute("DROP TABLE IF EXISTS villes")
connexion.execute("CREATE TABLE villes (nom TEXT, departement TEXT)")
for ville in villes:
    connexion.execute("INSERT INTO villes VALUES (?, ?)", ...)
connexion.commit()
print(len(connexion.execute("SELECT * FROM villes").fetchall()))
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_1_4.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "villes" in tables, "La table villes n'existe pas dans sorties/ex_1_4.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
lignes = v.execute("SELECT * FROM villes").fetchall()
v.close()
assert len(lignes) == 3, f"La table doit contenir 3 villes, elle en contient {len(lignes)}."
assert ("Nîmes", "Gard") in lignes, "Nîmes (Gard) doit être dans la table."
print("✅ OK")

### 1.5 La table des relevés

**Objectif** : Créer la table `releves` à six colonnes, avec les bons types.

**À faire** :
1. Complète le `CREATE TABLE` : `date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER`.

**Résultat attendu** :
```
['date', 'ville', 'temp_min', 'temp_max', 'pluie_mm', 'vent_kmh']
```

<details><summary>💡 Indice</summary>

Dans la chaîne, remplace `...` par `temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER`. (`PRAGMA table_info` est une requête spéciale qui décrit les colonnes : tu n'as pas à la retenir.)

📖 Cours, section 3.

</details>

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/ex_1_5.db")
connexion.execute("DROP TABLE IF EXISTS releves")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, ...)")
connexion.commit()
colonnes = [c[1] for c in connexion.execute("PRAGMA table_info(releves)").fetchall()]
print(colonnes)
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_1_5.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_1_5.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
infos = v.execute("PRAGMA table_info(releves)").fetchall()
v.close()
colonnes = [c[1] for c in infos]
types = [c[2] for c in infos]
assert colonnes == ["date", "ville", "temp_min", "temp_max", "pluie_mm", "vent_kmh"], f"Colonnes attendues : date, ville, temp_min, temp_max, pluie_mm, vent_kmh. Tu as {colonnes}."
assert types == ["TEXT", "TEXT", "REAL", "REAL", "REAL", "INTEGER"], f"Types attendus : TEXT, TEXT, REAL, REAL, REAL, INTEGER. Tu as {types}."
print("✅ OK")

### 1.6 Un relevé en base

**Objectif** : Ajouter un relevé complet (six valeurs).

**À faire** :
1. Ajoute le relevé de Montpellier du 2026-09-01 : 17.8, 29.2, 14.0 mm, 41 km/h.
2. Six `?` dans la requête, six valeurs dans le tuple. Puis `commit`.

**Résultat attendu** :
```
[('2026-09-01', 'Montpellier', 17.8, 29.2, 14.0, 41)]
```

<details><summary>💡 Indice</summary>

Le tuple complet : `("2026-09-01", "Montpellier", 17.8, 29.2, 14.0, 41)`.

📖 Cours, section 4.

</details>

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/ex_1_6.db")
connexion.execute("DROP TABLE IF EXISTS releves")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", ("2026-09-01", ...))
connexion.commit()
print(connexion.execute("SELECT * FROM releves").fetchall())
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_1_6.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_1_6.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
lignes = v.execute("SELECT * FROM releves").fetchall()
v.close()
assert lignes == [("2026-09-01", "Montpellier", 17.8, 29.2, 14.0, 41)], f"La table doit contenir le relevé de Montpellier, elle contient {lignes}. Si tu as eu 'Incorrect number of bindings', compte tes valeurs : il en faut 6."
print("✅ OK")

### 1.7 Tout le fichier en base

**Objectif** : Insérer les 180 relevés de `meteo.csv` avec une boucle.

**À faire** :
1. Pour chaque fiche `f`, complète le tuple de six valeurs : `f["date"]`, `f["ville"]`, … `f["vent_kmh"]`.
2. Un `commit` après la boucle.

**Résultat attendu** :
```
180
```

<details><summary>💡 Indice</summary>

Le tuple : `(f["date"], f["ville"], f["temp_min"], f["temp_max"], f["pluie_mm"], f["vent_kmh"])`.

📖 Cours, section 4, « Ranger tout le fichier ».

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = sqlite3.connect("sorties/ex_1_7.db")
connexion.execute("DROP TABLE IF EXISTS releves")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
for f in pd.read_csv("../donnees/meteo.csv").to_dict("records"):
    connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", (f["date"], f["ville"], ...))
connexion.commit()
print(len(connexion.execute("SELECT * FROM releves").fetchall()))
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_1_7.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_1_7.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
n = v.execute("SELECT COUNT(*) FROM releves").fetchall()[0][0]
premiere = v.execute("SELECT * FROM releves").fetchall()[:1]
v.close()
assert n == 180, f"La table doit contenir 180 relevés, elle en contient {n} : as-tu fait commit après la boucle ?"
assert premiere == [("2026-09-01", "Toulouse", 12.0, 25.9, 17.4, 11)], f"La première ligne est {premiere} : respecte l'ordre des colonnes."
print("✅ OK")

### 1.8 Tout lire

**Objectif** : Lire toutes les lignes avec `SELECT *` et `fetchall`.

**À faire** :
1. La base est prête (180 relevés).
2. Range dans `lignes` le résultat de `SELECT * FROM releves`, avec `fetchall()`.

**Résultat attendu** :
```
180 ('2026-09-01', 'Toulouse', 12.0, 25.9, 17.4, 11)
```

<details><summary>💡 Indice</summary>

`connexion.execute("SELECT * FROM releves").fetchall()`.

📖 Cours, section 5.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_8.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

lignes = connexion.execute(...).fetchall()
print(len(lignes), lignes[0])
connexion.close()

In [ ]:
# Vérification
assert "lignes" in dir() and lignes is not ..., "Exécute d'abord ta cellule."
assert isinstance(lignes, list), "lignes doit être une liste : as-tu ajouté .fetchall() ?"
assert len(lignes) == 180, f"lignes doit contenir 180 tuples, elle en contient {len(lignes)}."
print("✅ OK")

### 1.9 Les relevés d'une ville

**Objectif** : Filtrer avec `WHERE` et un `?`.

**À faire** :
1. Range dans `nimes` les colonnes `date` et `temp_max` des relevés de Nîmes.
2. Attention à la virgule du tuple à un élément : `("Nîmes",)`.

**Résultat attendu** :
```
30 ('2026-09-01', 29.2)
```

<details><summary>💡 Indice</summary>

`("Nîmes",)` : un tuple d'un seul élément, avec la virgule.

📖 Cours, sections 4 et 5.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_9.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

nimes = connexion.execute("SELECT date, temp_max FROM releves WHERE ville = ?", ...).fetchall()
print(len(nimes), nimes[0])
connexion.close()

In [ ]:
# Vérification
assert "nimes" in dir(), "La variable nimes n'existe pas : ta cellule a planté. Si l'erreur est 'Incorrect number of bindings … 5 supplied', ajoute la virgule : (\"Nîmes\",)."
assert len(nimes) == 30, f"nimes doit contenir 30 lignes, elle en contient {len(nimes)}."
assert nimes[0] == ("2026-09-01", 29.2), f"Chaque ligne doit être (date, temp_max) : la première est {nimes[0]}."
print("✅ OK")

### 1.10 Les journées très chaudes

**Objectif** : Filtrer avec une comparaison numérique.

**À faire** :
1. Range dans `tres_chaudes` les lignes (toutes colonnes) où `temp_max > ?`, avec la valeur 28.
2. Range leur nombre dans `nb_tres_chaudes`.

**Résultat attendu** :
```
15
```

<details><summary>💡 Indice</summary>

La condition dans la requête : `WHERE temp_max > ?`.

📖 Cours, section 5, « Conditions et tri ».

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_10.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

tres_chaudes = connexion.execute("SELECT * FROM releves WHERE ...", (28,)).fetchall()
nb_tres_chaudes = len(tres_chaudes)
print(nb_tres_chaudes)
connexion.close()

In [ ]:
# Vérification
assert "nb_tres_chaudes" in dir(), "La variable nb_tres_chaudes n'existe pas : ta cellule a planté. Lis la dernière ligne du message : 'syntax error' veut dire que la requête est mal écrite."
assert nb_tres_chaudes == 15, f"nb_tres_chaudes doit valoir 15, tu as {nb_tres_chaudes} : WHERE temp_max > ?"
print("✅ OK")

### 1.11 Trier avec ORDER BY

**Objectif** : Trier le résultat du plus grand au plus petit.

**À faire** :
1. Range dans `classement` les colonnes `ville`, `date`, `vent_kmh`, triées par `vent_kmh` décroissant (`ORDER BY vent_kmh DESC`).
2. Range dans `le_plus_venteux` la première ligne.

**Résultat attendu** :
```
('Nîmes', '2026-09-19', 55)
```

<details><summary>💡 Indice</summary>

`ORDER BY vent_kmh DESC`.

📖 Cours, section 5.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_11.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

classement = connexion.execute("SELECT ville, date, vent_kmh FROM releves ORDER BY ...").fetchall()
le_plus_venteux = classement[0]
print(le_plus_venteux)
connexion.close()

In [ ]:
# Vérification
assert "le_plus_venteux" in dir(), "La variable le_plus_venteux n'existe pas : ta cellule a planté."
assert le_plus_venteux[2] == 55, f"Le vent le plus fort est 55 km/h, tu as {le_plus_venteux[2]} : ORDER BY vent_kmh DESC (DESC = décroissant)."
print("✅ OK")

### 1.12 Corriger une valeur

**Objectif** : Modifier une case avec `UPDATE … SET … WHERE`.

**À faire** :
1. La station d'Albi signale que le 2026-09-03, la pluie était de 18.9 mm (et pas 18.2).
2. Corrige avec `UPDATE releves SET pluie_mm = ? WHERE ville = ? AND date = ?` et le tuple `(18.9, "Albi", "2026-09-03")`.
3. N'oublie pas `commit`.

**Résultat attendu** :
```
[(18.9,)]
```

<details><summary>💡 Indice</summary>

Le tuple : `(18.9, "Albi", "2026-09-03")`, dans l'ordre des `?`. Puis `connexion.commit()`.

📖 Cours, section 6.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_12.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

connexion.execute("UPDATE releves SET pluie_mm = ? WHERE ville = ? AND date = ?", ...)
...
print(connexion.execute("SELECT pluie_mm FROM releves WHERE ville = ? AND date = ?", ("Albi", "2026-09-03")).fetchall())
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_1_12.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_1_12.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
case = v.execute("SELECT pluie_mm FROM releves WHERE ville = 'Albi' AND date = '2026-09-03'").fetchall()
nb_189 = len(v.execute("SELECT * FROM releves WHERE pluie_mm = 18.9").fetchall())
v.close()
assert case == [(18.9,)], f"Pour une nouvelle connexion, la case vaut {case} : as-tu fait commit ?"
assert nb_189 == 1, f"{nb_189} lignes valent 18.9 mm : ton WHERE est trop large, une seule ligne doit changer."
print("✅ OK")

### 1.13 Supprimer une ville

**Objectif** : Supprimer des lignes avec `DELETE FROM … WHERE`.

**À faire** :
1. La station de Tarbes est fermée : supprime ses relevés (`DELETE FROM releves WHERE ville = ?`).
2. Valide avec `commit`.

**Résultat attendu** :
```
150
```

<details><summary>💡 Indice</summary>

`"DELETE FROM releves WHERE ville = ?"`.

📖 Cours, section 7.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_13.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

connexion.execute(..., ("Tarbes",))
connexion.commit()
print(len(connexion.execute("SELECT * FROM releves").fetchall()))
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_1_13.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_1_13.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
n = v.execute("SELECT COUNT(*) FROM releves").fetchall()[0][0]
tarbes = v.execute("SELECT COUNT(*) FROM releves WHERE ville = 'Tarbes'").fetchall()[0][0]
v.close()
assert n != 0, "La table est vide : as-tu oublié le WHERE ? Relance ta cellule, elle recrée la base."
assert tarbes == 0, f"Il reste {tarbes} relevés de Tarbes : DELETE FROM releves WHERE ville = ?, puis commit."
assert n == 150, f"Il doit rester 150 relevés, il en reste {n}."
print("✅ OK")

### 1.14 De la base à un DataFrame

**Objectif** : Récupérer une requête dans un DataFrame avec `pd.read_sql`.

**À faire** :
1. Range dans `tableau` le résultat de `pd.read_sql("SELECT * FROM releves", connexion)`.

**Résultat attendu** :
```
(180, 6)
```

<details><summary>💡 Indice</summary>

`pd.read_sql("SELECT * FROM releves", connexion)`.

📖 Cours, section 8.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_14.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

tableau = pd.read_sql(..., connexion)
connexion.close()
print(tableau.shape)

In [ ]:
# Vérification
assert "tableau" in dir() and tableau is not ..., "Exécute d'abord ta cellule."
assert tableau.shape == (180, 6), f"tableau doit faire (180, 6), il fait {tableau.shape}."
assert list(tableau.columns)[0] == "date", "Les colonnes doivent garder leurs noms : date, ville, ..."
print("✅ OK")

### 1.15 read_sql avec un paramètre

**Objectif** : Passer une valeur à `pd.read_sql` avec `params`.

**À faire** :
1. Range dans `tarbes` les relevés de Tarbes : `pd.read_sql("SELECT * FROM releves WHERE ville = ?", connexion, params=("Tarbes",))`.
2. Range dans `moyenne_tarbes` la moyenne de sa colonne `temp_max`, arrondie à 1 chiffre.

**Résultat attendu** :
```
20.6
```

<details><summary>💡 Indice</summary>

`params=("Tarbes",)`.

📖 Cours, section 8.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_15.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

tarbes = pd.read_sql("SELECT * FROM releves WHERE ville = ?", connexion, params=...)
connexion.close()
moyenne_tarbes = round(tarbes["temp_max"].mean(), 1)
print(moyenne_tarbes)

In [ ]:
# Vérification
assert "moyenne_tarbes" in dir(), "La variable moyenne_tarbes n'existe pas : ta cellule a planté. As-tu mis la virgule dans params=(\"Tarbes\",) ?"
assert len(tarbes) == 30, f"tarbes doit contenir 30 lignes, il en contient {len(tarbes)}."
assert moyenne_tarbes == 20.6, f"moyenne_tarbes doit valoir 20.6, tu as {moyenne_tarbes}."
print("✅ OK")

### 1.16 Réparer une requête dangereuse

**Objectif** : Remplacer une f-string par un `?`.

**À faire** :
1. La requête de la cellule utilise une f-string : elle plante avec `L'Isle-Jourdain`.
2. Réécris-la avec un `?` et le tuple `(ville,)`.
3. Range le résultat dans `trouves` (une liste vide est la bonne réponse : cette ville n'est pas en base).

**Résultat attendu** :
```
[]
```

<details><summary>💡 Indice</summary>

`connexion.execute("SELECT * FROM releves WHERE ville = ?", (ville,))`.

📖 Cours, section 4, « Pourquoi `?` ».

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_1_16.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

ville = "L'Isle-Jourdain"
# Version dangereuse : trouves = connexion.execute(f"SELECT * FROM releves WHERE ville = '{ville}'").fetchall()
trouves = connexion.execute(..., ...).fetchall()
print(trouves)
connexion.close()

In [ ]:
# Vérification
assert "trouves" in dir() and trouves is not ..., "La variable trouves n'existe pas : ta cellule a planté. Si l'erreur est 'syntax error', c'est que tu utilises encore une f-string."
assert trouves == [], f"trouves doit être une liste vide, tu as {trouves}."
print("✅ OK")

## Niveau 2 — Adapter

Le même geste dans un autre contexte, ou combiné avec un geste des modules précédents.

### 2.1 Un dictionnaire en table

**Objectif** : Insérer les couples d'un dictionnaire (module 06).

**À faire** :
1. Pour chaque `nom, departement` de `departements.items()`, fais un `INSERT` dans `villes`.
2. Puis range dans `noms` la liste des noms triés : `SELECT nom FROM villes ORDER BY nom`.

**Résultat attendu** :
```
[('Albi',), ('Montpellier',), ('Nîmes',), ('Perpignan',), ('Tarbes',), ('Toulouse',)]
```

<details><summary>💡 Indice</summary>

Dans la boucle : `connexion.execute("INSERT INTO villes VALUES (?, ?)", (nom, departement))`.

📖 Cours, sections 4 et 5 ; module 06 (`.items()`).

</details>

In [ ]:
import sqlite3

departements = {
    "Toulouse": "Haute-Garonne", "Montpellier": "Hérault", "Nîmes": "Gard",
    "Perpignan": "Pyrénées-Orientales", "Albi": "Tarn", "Tarbes": "Hautes-Pyrénées",
}
connexion = sqlite3.connect("sorties/ex_2_1.db")
connexion.execute("DROP TABLE IF EXISTS villes")
connexion.execute("CREATE TABLE villes (nom TEXT, departement TEXT)")
for nom, departement in departements.items():
    ...
connexion.commit()
noms = connexion.execute(...).fetchall()
print(noms)
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_2_1.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "villes" in tables, "La table villes n'existe pas dans sorties/ex_2_1.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
n = v.execute("SELECT COUNT(*) FROM villes").fetchall()[0][0]
v.close()
assert n == 6, f"La table doit contenir 6 villes, elle en contient {n} : un INSERT par couple, puis commit."
assert noms[0] == ("Albi",) and noms[-1] == ("Toulouse",), f"noms doit être trié par ordre alphabétique : ORDER BY nom. Tu as {noms}."
print("✅ OK")

### 2.2 Deux conditions et deux `?`

**Objectif** : Combiner deux conditions avec `AND`, avec deux valeurs paramétrées.

**À faire** :
1. Range dans `jours` les dates et pluies de Perpignan où `pluie_mm > 5`, triées par pluie décroissante.
2. Deux `?` dans la requête, donc deux valeurs dans le tuple, dans l'ordre.

**Résultat attendu** :
```
10 ('2026-09-14', 19.9)
```

<details><summary>💡 Indice</summary>

`WHERE ville = ? AND pluie_mm > ? ORDER BY pluie_mm DESC` et le tuple `("Perpignan", 5)`.

📖 Cours, section 5.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_2_2.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

requete = "SELECT date, pluie_mm FROM releves WHERE ville = ? AND ... ORDER BY ..."
jours = connexion.execute(requete, (...)).fetchall()
print(len(jours), jours[0])
connexion.close()

In [ ]:
# Vérification
assert "jours" in dir(), "La variable jours n'existe pas : ta cellule a planté (lis la dernière ligne du message)."
assert len(jours) == 10, f"jours doit contenir 10 lignes, il en contient {len(jours)} : WHERE ville = ? AND pluie_mm > ?, avec (\"Perpignan\", 5)."
assert jours[0][1] == 19.9, f"La première ligne doit être la plus pluvieuse (19.9 mm) : ORDER BY pluie_mm DESC."
print("✅ OK")

### 2.3 Les trois nuits les plus froides

**Objectif** : Trier en ordre croissant, puis garder les 3 premières lignes avec une tranche (module 05).

**À faire** :
1. Range dans `nuits` les colonnes `ville`, `date`, `temp_min` triées par `temp_min` croissante (sans `DESC`).
2. Range dans `top3` les trois premières : `nuits[:3]`.

**Résultat attendu** :
```
[('Tarbes', '2026-09-29', 4.9), ('Tarbes', '2026-09-26', 5.1), ('Tarbes', '2026-09-27', 5.5)]
```

<details><summary>💡 Indice</summary>

`ORDER BY temp_min` trie du plus petit au plus grand.

📖 Cours, section 5 ; module 05 (tranches).

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_2_3.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

nuits = connexion.execute(...).fetchall()
top3 = ...
print(top3)
connexion.close()

In [ ]:
# Vérification
assert "top3" in dir() and top3 is not ..., "Exécute d'abord ta cellule."
assert len(top3) == 3, f"top3 doit contenir 3 lignes, il en contient {len(top3)} : nuits[:3]."
assert top3[0][2] == 4.9, f"La nuit la plus froide est à 4.9 °C, tu as {top3[0][2]} : ORDER BY temp_min (croissant, sans DESC)."
print("✅ OK")

### 2.4 Une fonction de recherche

**Objectif** : Ranger une requête paramétrée dans une fonction (module 07).

**À faire** :
1. Complète `releves_de(connexion, ville)` : elle renvoie la liste des `(date, temp_max)` de la ville demandée.
2. Range dans `n_albi` et `n_inconnue` le nombre de lignes pour `"Albi"` et pour `"Carcassonne"`.

**Résultat attendu** :
```
30 0
```

<details><summary>💡 Indice</summary>

Dans la fonction : `connexion.execute("SELECT date, temp_max FROM releves WHERE ville = ?", (ville,))`.

📖 Cours, section 5 ; module 07.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_2_4.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)


def releves_de(connexion, ville):
    return connexion.execute(...).fetchall()


n_albi = len(releves_de(connexion, "Albi"))
n_inconnue = len(releves_de(connexion, "Carcassonne"))
print(n_albi, n_inconnue)
connexion.close()

In [ ]:
# Vérification
assert "n_albi" in dir() and "n_inconnue" in dir(), "Exécute d'abord ta cellule (elle a peut-être planté : lis la dernière ligne du message)."
assert n_albi == 30, f"releves_de(connexion, \"Albi\") doit renvoyer 30 lignes, elle en renvoie {n_albi}. Utilise le paramètre ville avec un ?, pas \"Albi\" écrit en dur."
assert n_inconnue == 0, f"Pour Carcassonne, il faut 0 ligne, tu en as {n_inconnue}."
print("✅ OK")

### 2.5 Le fichier brut, nettoyé puis rangé

**Objectif** : Nettoyer `meteo_brut.csv` avec pandas (module 10), puis l'insérer en base.

**À faire** :
1. Le nettoyage pandas est écrit pour toi (villes, virgules, doublons, cases vides).
2. Insère chaque fiche de `propre.to_dict("records")` dans la table, puis `commit`.
3. Range dans `nb_en_base` le nombre de lignes en base.

**Résultat attendu** :
```
166
```

<details><summary>💡 Indice</summary>

Dans la boucle : `connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", (f["date"], f["ville"], f["temp_min"], f["temp_max"], f["pluie_mm"], f["vent_kmh"]))`.

📖 Cours, section 4 ; module 10, section 9.

</details>

In [ ]:
import sqlite3
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
corrections = {"toulouse": "Toulouse", "montpellier": "Montpellier", "nîmes": "Nîmes",
               "perpignan": "Perpignan", "albi": "Albi", "tarbes": "Tarbes"}
brut["ville"] = brut["ville"].str.strip().replace(corrections)
brut["temp_max"] = pd.to_numeric(brut["temp_max"].str.replace(",", "."))
propre = brut.drop_duplicates().dropna()

connexion = sqlite3.connect("sorties/ex_2_5.db")
connexion.execute("DROP TABLE IF EXISTS releves")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
for f in propre.to_dict("records"):
    ...
connexion.commit()
nb_en_base = len(connexion.execute("SELECT * FROM releves").fetchall())
print(nb_en_base)
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_2_5.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_2_5.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
n = v.execute("SELECT COUNT(*) FROM releves").fetchall()[0][0]
villes = v.execute("SELECT DISTINCT ville FROM releves").fetchall()
v.close()
assert n == 166, f"La base doit contenir 166 relevés propres, elle en contient {n}."
assert len(villes) == 6, f"La base doit contenir 6 villes bien écrites, elle en contient {len(villes)}."
print("✅ OK")

### 2.6 Corriger toute une station

**Objectif** : Un `UPDATE` qui calcule à partir de l'ancienne valeur.

**À faire** :
1. Le capteur de Tarbes sous-estimait de 0.5 °C : ajoute 0.5 à toutes ses `temp_max`.
2. En SQL, `SET temp_max = temp_max + ?` utilise l'ancienne valeur de la case. Le `WHERE` limite à Tarbes.
3. `commit`, puis range dans `nouvelle_moyenne` la moyenne de `temp_max` de Tarbes (avec `pd.read_sql`), arrondie à 1 chiffre.

**Résultat attendu** :
```
21.1
```

<details><summary>💡 Indice</summary>

`WHERE ville = ?`.

📖 Cours, section 6.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_2_6.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

connexion.execute("UPDATE releves SET temp_max = temp_max + ? WHERE ...", (0.5, "Tarbes"))
connexion.commit()
tarbes = pd.read_sql("SELECT temp_max FROM releves WHERE ville = ?", connexion, params=("Tarbes",))
connexion.close()
nouvelle_moyenne = round(tarbes["temp_max"].mean(), 1)
print(nouvelle_moyenne)

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_2_6.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_2_6.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
assert "nouvelle_moyenne" in dir() and nouvelle_moyenne is not ..., "La variable nouvelle_moyenne n'existe pas : ta cellule a planté (lis la dernière ligne du message ; 'syntax error' = requête incomplète)."
albi = v.execute("SELECT temp_max FROM releves WHERE ville = 'Albi' AND date = '2026-09-01'").fetchall()
v.close()
assert albi == [(26.8,)], "Albi a aussi changé : ton WHERE doit limiter la correction à Tarbes (WHERE ville = ?)."
assert nouvelle_moyenne == 21.1, f"nouvelle_moyenne doit valoir 21.1, tu as {nouvelle_moyenne}."
print("✅ OK")

### 2.7 Supprimer les mesures suspectes

**Objectif** : Un `DELETE` avec une condition numérique.

**À faire** :
1. Les mesures de vent au-dessus de 50 km/h sont jugées suspectes : supprime ces lignes (`vent_kmh > ?`).
2. `commit`, puis range dans `nb_restants` le nombre de lignes restantes.

**Résultat attendu** :
```
165
```

<details><summary>💡 Indice</summary>

`"DELETE FROM releves WHERE vent_kmh > ?"`.

📖 Cours, section 7.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_2_7.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

connexion.execute(..., (50,))
connexion.commit()
nb_restants = len(connexion.execute("SELECT * FROM releves").fetchall())
print(nb_restants)
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_2_7.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_2_7.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
n = v.execute("SELECT COUNT(*) FROM releves").fetchall()[0][0]
v.close()
assert n != 0, "La table est vide : il manque le WHERE. Relance ta cellule, elle recrée la base."
assert n == 165, f"Il doit rester 165 relevés, il en reste {n} : DELETE FROM releves WHERE vent_kmh > ?"
print("✅ OK")

### 2.8 La base, puis groupby

**Objectif** : Lire la base avec `pd.read_sql`, puis calculer par ville avec pandas (module 10).

**À faire** :
1. Range dans `vents` les colonnes `ville` et `vent_kmh` de toute la table (`pd.read_sql`).
2. Range dans `vent_moyen` la moyenne de `vent_kmh` par ville, arrondie avec `.round(1)`.

**Résultat attendu** :
```
ville
Albi           29.5
Montpellier    36.3
Nîmes          28.9
Perpignan      35.0
Tarbes         29.3
Toulouse       27.4
Name: vent_kmh, dtype: float64
```

<details><summary>💡 Indice</summary>

`pd.read_sql("SELECT ville, vent_kmh FROM releves", connexion)` puis `vents.groupby("ville")["vent_kmh"].mean().round(1)`.

📖 Cours, section 8 ; module 10, section 8.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_2_8.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

vents = pd.read_sql(..., connexion)
connexion.close()
vent_moyen = ...
print(vent_moyen)

In [ ]:
# Vérification
assert "vent_moyen" in dir() and vent_moyen is not ..., "Exécute d'abord ta cellule."
assert list(vents.columns) == ["ville", "vent_kmh"], f"vents doit avoir les colonnes ville et vent_kmh, il a {list(vents.columns)}."
assert len(vent_moyen) == 6, "vent_moyen doit avoir une valeur par ville : groupby(\"ville\")."
assert vent_moyen["Albi"] == 29.5, f"Le vent moyen d'Albi doit valoir 29.5, tu as {vent_moyen['Albi']}."
print("✅ OK")

### 2.9 Des objets en base

**Objectif** : Insérer des objets (module 11) en lisant leurs attributs.

**À faire** :
1. `previsions` est une liste d'objets `Prevision` (dataclass).
2. Pour chaque `p`, insère `(p.date, p.temp_min, p.temp_max)` dans la table `previsions`.
3. `commit`.

**Résultat attendu** :
```
[('2026-10-01', 11.2, 22.1), ('2026-10-02', 12.0, 21.4), ('2026-10-03', 10.5, 19.8)]
```

<details><summary>💡 Indice</summary>

Le tuple des attributs : `(p.date, p.temp_min, p.temp_max)`.

📖 Cours, section 4 ; module 11, section 8.

</details>

In [ ]:
import sqlite3
from dataclasses import dataclass


@dataclass
class Prevision:
    date: str
    temp_min: float
    temp_max: float


previsions = [Prevision("2026-10-01", 11.2, 22.1), Prevision("2026-10-02", 12.0, 21.4), Prevision("2026-10-03", 10.5, 19.8)]

connexion = sqlite3.connect("sorties/ex_2_9.db")
connexion.execute("DROP TABLE IF EXISTS previsions")
connexion.execute("CREATE TABLE previsions (date TEXT, temp_min REAL, temp_max REAL)")
for p in previsions:
    connexion.execute("INSERT INTO previsions VALUES (?, ?, ?)", ...)
connexion.commit()
print(connexion.execute("SELECT * FROM previsions").fetchall())
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_2_9.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "previsions" in tables, "La table previsions n'existe pas dans sorties/ex_2_9.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
lignes = v.execute("SELECT * FROM previsions").fetchall()
v.close()
assert len(lignes) == 3, f"La table doit contenir 3 prévisions, elle en contient {len(lignes)}."
assert lignes[0] == ("2026-10-01", 11.2, 22.1), f"La première ligne doit être ('2026-10-01', 11.2, 22.1), elle est {lignes[0]}."
print("✅ OK")

### 2.10 De la base aux objets

**Objectif** : Transformer les tuples de `fetchall` en objets (module 11), avec le déballage (module 06).

**À faire** :
1. Pour chaque ligne `(date, ville, temp_max)` des relevés de Montpellier, déballe-la puis fabrique un `Releve`.
2. Range les objets dans `objets`, puis le relevé le plus chaud (le premier, puisque la requête trie) dans `record`.

**Résultat attendu** :
```
29.2
```

<details><summary>💡 Indice</summary>

`date, ville, temp_max = ligne` puis `objets.append(Releve(date, ville, temp_max))`.

📖 Cours, section 5 ; module 06 (déballage) ; module 11.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_2_10.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)
from dataclasses import dataclass


@dataclass
class Releve:
    date: str
    ville: str
    temp_max: float


lignes = connexion.execute(
    "SELECT date, ville, temp_max FROM releves WHERE ville = ? ORDER BY temp_max DESC", ("Montpellier",)
).fetchall()
connexion.close()

objets = []
for ligne in lignes:
    date, ville, temp_max = ...
    objets.append(...)
record = objets[0]
print(record.temp_max)

In [ ]:
# Vérification
assert "objets" in dir() and "record" in dir(), "Exécute d'abord ta cellule (elle a peut-être planté sur un ...)."
assert len(objets) == 30, f"objets doit contenir 30 relevés, il en contient {len(objets)}."
assert record.ville == "Montpellier" and record.temp_max == 29.2, f"record doit être le relevé le plus chaud de Montpellier (29.2 °C), tu as {record}."
print("✅ OK")

### 2.11 Compter avec len

**Objectif** : Compter les lignes renvoyées par une requête.

**À faire** :
1. Range dans `nb_jours_pluie` le nombre de jours de pluie (`pluie_mm > 0`) à Perpignan : une requête à deux `?`, puis `len` de la liste.

**Résultat attendu** :
```
21
```

<details><summary>💡 Indice</summary>

`len(connexion.execute("SELECT date FROM releves WHERE ville = ? AND pluie_mm > ?", ("Perpignan", 0)).fetchall())`.

📖 Cours, section 5.

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_2_11.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

nb_jours_pluie = ...
print(nb_jours_pluie)
connexion.close()

In [ ]:
# Vérification
assert "nb_jours_pluie" in dir() and nb_jours_pluie is not ..., "Exécute d'abord ta cellule."
assert nb_jours_pluie != 30, "30 : c'est le nombre total de jours de Perpignan. Ajoute la condition AND pluie_mm > ?."
assert nb_jours_pluie == 21, f"nb_jours_pluie doit valoir 21, tu as {nb_jours_pluie}."
print("✅ OK")

## Niveau 3 — Transposer

Un petit problème décrit en français. À toi de choisir les gestes. Seuls les noms imposés comptent.

### 3.1 La table des stations

**Objectif** : Créer dans `sorties/ex_3_1.db` une table `stations` (nom texte, departement texte, altitude entier), y ranger les 6 stations de la liste `donnees`, puis trouver les stations à plus de 100 m d'altitude, de la plus haute à la plus basse.

**À faire** :
1. Range dans `hautes` la liste des **noms** (des chaînes, pas des tuples) des stations à plus de 100 m, triées par altitude décroissante.

**Résultat attendu** :
```
['Tarbes', 'Albi', 'Toulouse']
```

<details><summary>💡 Indice 1 : la méthode</summary>

Tous les gestes du cours dans l'ordre : `DROP` + `CREATE` (section 3), une boucle d'`INSERT` paramétrés puis `commit` (section 4), un `SELECT` avec une condition `>` et un tri décroissant (section 5). Chaque ligne de `fetchall` est un tuple : prends sa position 0.

📖 Cours, sections 3, 4 et 5.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
connexion = sqlite3.connect("sorties/ex_3_1.db")
connexion.execute("DROP TABLE IF EXISTS stations")
connexion.execute("CREATE TABLE stations (nom TEXT, departement TEXT, altitude INTEGER)")
for station in donnees:
    connexion.execute("INSERT INTO stations VALUES (?, ?, ?)", station)
connexion.commit()
lignes = connexion.execute("SELECT nom FROM stations WHERE altitude > ? ORDER BY altitude DESC", (100,)).fetchall()
hautes = [ligne[0] for ligne in lignes]
connexion.close()
```

</details>

In [ ]:
import sqlite3

donnees = [('Toulouse', 'Haute-Garonne', 146), ('Montpellier', 'Hérault', 27), ('Nîmes', 'Gard', 39), ('Perpignan', 'Pyrénées-Orientales', 37), ('Albi', 'Tarn', 169), ('Tarbes', 'Hautes-Pyrénées', 304)]
# À toi de jouer, étape par étape :
# 1. Ouvre la base et crée la table stations à neuf (trois colonnes, avec leurs types).
# 2. Range les 6 stations, puis valide.
# 3. Demande les noms des stations à plus de 100 m, de la plus haute à la plus basse.
# 4. Transforme la liste de tuples en liste de noms, puis ferme.

hautes = ...
print(hautes)

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_3_1.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "stations" in tables, "La table stations n'existe pas dans sorties/ex_3_1.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
n = v.execute("SELECT COUNT(*) FROM stations").fetchall()[0][0]
v.close()
assert n == 6, f"La table stations doit contenir 6 lignes (validées par commit), elle en contient {n}."
assert hautes == ["Tarbes", "Albi", "Toulouse"], f"hautes doit valoir ['Tarbes', 'Albi', 'Toulouse'], tu as {hautes}. Chaque ligne de fetchall est un tuple : prends ligne[0]."
print("✅ OK")

### 3.2 Une fonction d'ajout

**Objectif** : Écrire `ajouter_releve(connexion, fiche)` : elle insère un dictionnaire de relevé (6 clés) avec une requête paramétrée **et** valide. L'utiliser pour ajouter les 3 fiches de `nouveaux` à une base neuve `sorties/ex_3_2.db`.

**À faire** :
1. Range dans `nb_total` le nombre de lignes de la table après les 3 ajouts.

**Résultat attendu** :
```
3
```

<details><summary>💡 Indice 1 : la méthode</summary>

Le corps de la fonction, c'est l'`INSERT` à six `?` de la section 4, avec les six valeurs lues dans le dictionnaire `fiche`, suivi de `commit` (sinon une nouvelle connexion ne voit rien).

📖 Cours, section 4 ; module 07.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
def ajouter_releve(connexion, fiche):
    connexion.execute(
        "INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)",
        (fiche["date"], fiche["ville"], fiche["temp_min"], fiche["temp_max"], fiche["pluie_mm"], fiche["vent_kmh"]),
    )
    connexion.commit()


for fiche in nouveaux:
    ajouter_releve(connexion, fiche)
nb_total = len(connexion.execute("SELECT * FROM releves").fetchall())
```

</details>

In [ ]:
import sqlite3

nouveaux = [
    {"date": "2026-10-01", "ville": "Toulouse", "temp_min": 11.2, "temp_max": 22.1, "pluie_mm": 0.0, "vent_kmh": 14},
    {"date": "2026-10-02", "ville": "Toulouse", "temp_min": 12.0, "temp_max": 21.4, "pluie_mm": 0.0, "vent_kmh": 9},
    {"date": "2026-10-03", "ville": "Toulouse", "temp_min": 10.5, "temp_max": 19.8, "pluie_mm": 4.2, "vent_kmh": 22},
]
connexion = sqlite3.connect("sorties/ex_3_2.db")
connexion.execute("DROP TABLE IF EXISTS releves")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
# À toi de jouer, étape par étape :
# 1. Écris la fonction ajouter_releve : une insertion paramétrée, puis la validation.
# 2. Utilise-la pour chacune des 3 fiches.
# 3. Compte les lignes de la table.

nb_total = ...
print(nb_total)
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_3_2.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_3_2.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
lignes = v.execute("SELECT * FROM releves").fetchall()
v.close()
assert "ajouter_releve" in dir(), "La fonction ajouter_releve n'existe pas."
assert len(lignes) == 3, f"Une nouvelle connexion voit {len(lignes)} lignes au lieu de 3 : ta fonction fait-elle commit ?"
assert lignes[2] == ("2026-10-03", "Toulouse", 10.5, 19.8, 4.2, 22), f"La 3e ligne ne correspond pas : {lignes[2]}."
assert nb_total == 3, f"nb_total doit valoir 3, tu as {nb_total}."
print("✅ OK")

### 3.3 Une recherche qui ne plante jamais

**Objectif** : Écrire `chercher(connexion, ville)` qui renvoie le **nombre** de relevés d'une ville. Elle doit marcher pour `"Albi"`, pour `"L'Isle-Jourdain"` (apostrophe) et pour le piège `"x' OR '1'='1"`, sans erreur et sans renvoyer toute la table.

**À faire** :
1. Range dans `resultats` la liste des trois réponses, dans l'ordre : Albi, L'Isle-Jourdain, le piège.

**Résultat attendu** :
```
[30, 0, 0]
```

<details><summary>💡 Indice 1 : la méthode</summary>

Une seule ligne dans la fonction : un `SELECT … WHERE ville = ?` avec un tuple à un élément (section 4, « Pourquoi `?` »), et `len` de la liste rapportée.

📖 Cours, section 4.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
def chercher(connexion, ville):
    return len(connexion.execute("SELECT * FROM releves WHERE ville = ?", (ville,)).fetchall())


resultats = [chercher(connexion, "Albi"), chercher(connexion, "L'Isle-Jourdain"), chercher(connexion, "x' OR '1'='1")]
```

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_3_3.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

# À toi de jouer, étape par étape :
# 1. Écris la fonction chercher : une requête paramétrée qui renvoie le nombre de lignes trouvées.
# 2. Range dans resultats les réponses pour les trois noms, dans l'ordre.

resultats = ...
print(resultats)
connexion.close()

In [ ]:
# Vérification
assert "chercher" in dir(), "La fonction chercher n'existe pas."
assert "resultats" in dir() and resultats is not ..., "Exécute d'abord ta cellule."
assert resultats != [30, 0, 180], "Le piège renvoie toute la table (180) : tu as utilisé une f-string. Utilise un ? et (ville,)."
assert resultats == [30, 0, 0], f"resultats doit valoir [30, 0, 0], tu as {resultats}."
print("✅ OK")

### 3.4 Nettoyer directement en base

**Objectif** : Le fichier brut a été rangé **tel quel** en base (villes avec espaces et minuscules). Répare la colonne `ville` avec des `UPDATE` : pour chaque nom abîmé de la liste `abimes`, mets le bon nom.

**À faire** :
1. Pour chaque nom de `abimes`, le bon nom s'obtient avec `.strip()` puis le dictionnaire `corrections` (module 02 et 06). Si le nom nettoyé n'est pas dans `corrections`, il est déjà bon.
2. Valide, puis range dans `villes_en_base` l'ensemble (`set`) des villes de la table.

**Résultat attendu** :
```
{'Albi', 'Montpellier', 'Nîmes', 'Perpignan', 'Tarbes', 'Toulouse'} (dans un ordre quelconque)
```

<details><summary>💡 Indice 1 : la méthode</summary>

Le bon nom : `.strip()` (module 02) puis le dictionnaire `corrections` ; si le nom nettoyé n'est pas une clé, il est déjà bon (la méthode `.get` avec une valeur par défaut, module 06). La correction : un `UPDATE … SET … WHERE` à deux `?` (section 6). L'ensemble : un `set` vide qu'on remplit avec `.add` (module 06).

📖 Cours, section 6.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
for ancien in abimes:
    nouveau = ancien.strip()
    nouveau = corrections.get(nouveau, nouveau)
    connexion.execute("UPDATE releves SET ville = ? WHERE ville = ?", (nouveau, ancien))
connexion.commit()

villes_en_base = set()
for ligne in connexion.execute("SELECT ville FROM releves").fetchall():
    villes_en_base.add(ligne[0])
```

</details>

In [ ]:
import sqlite3
import pandas as pd

brut = pd.read_csv("../donnees/meteo_brut.csv", sep=";")
connexion = sqlite3.connect("sorties/ex_3_4.db")
connexion.execute("DROP TABLE IF EXISTS releves")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_max TEXT)")
for f in brut.to_dict("records"):
    connexion.execute("INSERT INTO releves VALUES (?, ?, ?)", (f["date"], f["ville"], f["temp_max"]))
connexion.commit()

corrections = {"toulouse": "Toulouse", "montpellier": "Montpellier", "nîmes": "Nîmes",
               "perpignan": "Perpignan", "albi": "Albi", "tarbes": "Tarbes"}
abimes = sorted(set(brut["ville"]))
# À toi de jouer, étape par étape :
# 1. Pour chaque nom abîmé : calcule le bon nom, puis corrige la table.
# 2. Valide.
# 3. Lis toutes les villes de la table et range-les dans un ensemble.

villes_en_base = ...
print(villes_en_base)
connexion.close()

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/ex_3_4.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/ex_3_4.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
noms = set(l[0] for l in v.execute("SELECT ville FROM releves").fetchall())
n = v.execute("SELECT COUNT(*) FROM releves").fetchall()[0][0]
v.close()
attendu = {"Albi", "Montpellier", "Nîmes", "Perpignan", "Tarbes", "Toulouse"}
assert n == 185, f"La table doit garder ses 185 lignes (on corrige, on ne supprime pas), elle en a {n}."
assert noms == attendu, f"Pour une nouvelle connexion, les villes sont {noms} : as-tu fait commit après les UPDATE ?"
assert villes_en_base == attendu, f"villes_en_base doit valoir {attendu}, tu as {villes_en_base}."
print("✅ OK")

### 3.5 Le podium de la pluie

**Objectif** : Inès veut les 3 journées les plus pluvieuses de Toulouse, en phrases : `"2026-09-01 : 17.4 mm"`.

**À faire** :
1. Range les 3 phrases, de la plus pluvieuse à la moins pluvieuse, dans la liste `podium`.

**Résultat attendu** :
```
2026-09-01 : 17.4 mm
2026-09-17 : 16.1 mm
2026-09-29 : 11.6 mm
```

<details><summary>💡 Indice 1 : la méthode</summary>

Un `SELECT … WHERE … ORDER BY … DESC` (section 5), une tranche des 3 premières lignes (module 05), une boucle avec déballage de chaque tuple et une f-string.

📖 Cours, section 5.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
lignes = connexion.execute(
    "SELECT date, pluie_mm FROM releves WHERE ville = ? ORDER BY pluie_mm DESC", ("Toulouse",)
).fetchall()
podium = []
for date, pluie in lignes[:3]:
    podium.append(f"{date} : {pluie} mm")
```

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_3_5.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

# À toi de jouer, étape par étape :
# 1. Demande les dates et les pluies de Toulouse, de la plus pluvieuse à la moins pluvieuse.
# 2. Garde les 3 premières lignes et fabrique une phrase pour chacune.

podium = ...
for phrase in podium:
    print(phrase)
connexion.close()

In [ ]:
# Vérification
assert "podium" in dir() and podium is not ..., "Exécute d'abord ta cellule."
assert len(podium) == 3, f"podium doit contenir 3 phrases, il en contient {len(podium)}."
assert podium == ['2026-09-01 : 17.4 mm', '2026-09-17 : 16.1 mm', '2026-09-29 : 11.6 mm'], f"podium doit valoir ['2026-09-01 : 17.4 mm', '2026-09-17 : 16.1 mm', '2026-09-29 : 11.6 mm'], tu as {podium}."
print("✅ OK")

### 3.6 Le rapport pluie

**Objectif** : Trouver la ville la plus arrosée du mois (avec `pd.read_sql` + `groupby`), puis son jour record (avec une requête SQL), et écrire la phrase : `"Perpignan : 136.3 mm en septembre, record le 2026-09-14 (19.9 mm)"`.

**À faire** :
1. Range la ville dans `ville_arrosee` et la phrase complète dans `phrase` (cumul arrondi à 1 chiffre).

**Résultat attendu** :
```
Perpignan : 136.3 mm en septembre, record le 2026-09-14 (19.9 mm)
```

<details><summary>💡 Indice 1 : la méthode</summary>

Deux temps : pandas pour le cumul par ville (`pd.read_sql`, section 8, puis le regroupement et le tri du module 10), puis SQL pour le jour record (un tri décroissant, la première ligne, déballée en deux variables).

📖 Cours, sections 5 et 8.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
tout = pd.read_sql("SELECT ville, pluie_mm FROM releves", connexion)
cumul = tout.groupby("ville")["pluie_mm"].sum().round(1).sort_values(ascending=False)
ville_arrosee = list(cumul.index)[0]
lignes = connexion.execute(
    "SELECT date, pluie_mm FROM releves WHERE ville = ? ORDER BY pluie_mm DESC", (ville_arrosee,)
).fetchall()
date_record, pluie_record = lignes[0]
phrase = f"{ville_arrosee} : {cumul[ville_arrosee]} mm en septembre, record le {date_record} ({pluie_record} mm)"
```

</details>

In [ ]:
import sqlite3
import pandas as pd

connexion = preparer_base("sorties/ex_3_6.db")   # base neuve avec les 180 relevés (cellule de préparation, en haut)

# À toi de jouer, étape par étape :
# 1. Lis la ville et la pluie de toute la table dans un DataFrame.
# 2. Calcule le cumul de pluie par ville et trouve la ville la plus arrosée.
# 3. Demande à la base le jour le plus pluvieux de cette ville.
# 4. Construis la phrase.

ville_arrosee = ...
phrase = ...
print(phrase)
connexion.close()

In [ ]:
# Vérification
assert "phrase" in dir() and phrase is not ..., "Exécute d'abord ta cellule."
assert ville_arrosee == "Perpignan", f"ville_arrosee doit valoir Perpignan, tu as {ville_arrosee}."
assert phrase == "Perpignan : 136.3 mm en septembre, record le 2026-09-14 (19.9 mm)", f"phrase attendue : 'Perpignan : 136.3 mm en septembre, record le 2026-09-14 (19.9 mm)'. Tu as : '{phrase}'"
print("✅ OK")

## 🔁 Révisions espacées

Cinq exercices sur les modules précédents, **sans base de données**.

### R.1 (module 11) Une méthode de station

**Objectif** : Écrire une méthode qui parcourt les relevés d'une station.

**À faire** :
1. Complète `pluie_totale` : la somme des `pluie_mm` des relevés, arrondie à 1 chiffre.
2. Range le résultat pour `albi` dans `total_albi`.

**Résultat attendu** :
```
32.6
```

<details><summary>💡 Indice</summary>

`total = total + releve.pluie_mm`.

📖 Module 11, cours, section 7.

</details>

In [ ]:
from dataclasses import dataclass


@dataclass
class Releve:
    date: str
    pluie_mm: float


class Station:
    def __init__(self, nom):
        self.nom = nom
        self.releves = []

    def ajouter(self, releve):
        self.releves.append(releve)

    def pluie_totale(self):
        total = 0
        for releve in self.releves:
            total = total + ...
        return round(total, 1)


albi = Station("Albi")
for date, pluie in [("2026-09-01", 9.4), ("2026-09-02", 5.0), ("2026-09-03", 18.2)]:
    albi.ajouter(Releve(date, pluie))
total_albi = albi.pluie_totale()
print(total_albi)

In [ ]:
# Vérification
assert "total_albi" in dir(), "La variable total_albi n'existe pas : ta cellule a planté. Si le message parle de 'ellipsis', remplace le ... par releve.pluie_mm."
assert total_albi == 32.6, f"total_albi doit valoir 32.6, tu as {total_albi}."
print("✅ OK")

### R.2 (module 11) fit puis predict

**Objectif** : Utiliser un modèle avec son mode d'emploi.

**À faire** :
1. Entraîne `modele` sur `septembre` avec `fit`, puis range dans `semaine` la prévision pour 7 jours.

**Résultat attendu** :
```
[24.0, 24.0, 24.0, 24.0, 24.0, 24.0, 24.0]
```

<details><summary>💡 Indice</summary>

`modele.fit(septembre)` puis `semaine = modele.predict(7)`.

📖 Module 11, cours, section 9.

</details>

In [ ]:
class ModeleMoyenne:
    def __init__(self):
        self.moyenne = None

    def fit(self, valeurs):
        self.moyenne = round(sum(valeurs) / len(valeurs), 1)

    def predict(self, nb_jours):
        previsions = []
        for jour in range(nb_jours):
            previsions.append(self.moyenne)
        return previsions


septembre = [25.9, 23.2, 27.2, 19.7]
modele = ModeleMoyenne()
...
semaine = ...
print(semaine)

In [ ]:
# Vérification
assert "semaine" in dir() and semaine is not ..., "Exécute d'abord ta cellule."
assert semaine != [None] * 7, "Que des None : le modèle n'a rien appris. Appelle modele.fit(septembre) AVANT predict."
assert semaine == [24.0] * 7, f"semaine doit valoir sept fois 24.0, tu as {semaine}."
print("✅ OK")

### R.3 (module 10) Un filtre pandas

**Objectif** : Filtrer avec deux conditions en pandas (attention : `&` ici, pas `AND`).

**À faire** :
1. Range dans `nb_mistral` le nombre de jours à Nîmes où `vent_kmh > 40`.

**Résultat attendu** :
```
7
```

<details><summary>💡 Indice</summary>

Deuxième condition : `(meteo["vent_kmh"] > 40)`. En pandas `&` ; en SQL `AND`.

📖 Module 10, cours, section 6.

</details>

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
venteux = meteo[(meteo["ville"] == "Nîmes") & (...)]
nb_mistral = len(venteux)
print(nb_mistral)

In [ ]:
# Vérification
assert "nb_mistral" in dir(), "Exécute d'abord ta cellule."
assert nb_mistral == 7, f"nb_mistral doit valoir 7, tu as {nb_mistral} : (meteo[\"vent_kmh\"] > 40)."
print("✅ OK")

### R.4 (module 10) Un graphique enregistré

**Objectif** : Tracer et enregistrer un graphique en barres.

**À faire** :
1. Trace les barres du cumul de pluie par ville, avec un titre.
2. Enregistre dans `sorties/r4_pluie.png` avant `plt.show()`.

**Résultat attendu** :
```
Six barres.
```

<details><summary>💡 Indice</summary>

`plt.bar(list(cumul.index), list(cumul))`.

📖 Module 10, cours, section 10.

</details>

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

Path("sorties").mkdir(exist_ok=True)
meteo = pd.read_csv("../donnees/meteo.csv")
cumul = meteo.groupby("ville")["pluie_mm"].sum()
plt.bar(..., ...)
plt.title("Cumul de pluie, septembre 2026")
plt.savefig(...)
plt.show()

In [ ]:
# Vérification
from pathlib import Path
assert Path("sorties/r4_pluie.png").exists(), "L'image sorties/r4_pluie.png n'existe pas : plt.savefig(\"sorties/r4_pluie.png\")."
print("✅ OK")

### R.5 (module 04) Une boucle while

**Objectif** : Combien de jours faut-il à Albi pour cumuler au moins 50 mm de pluie ?

**À faire** :
1. Tant que `cumul` est inférieur à 50, ajoute la pluie du jour `pluies[jour]` à `cumul`, puis passe au jour suivant (`jour = jour + 1`).
2. À la fin, `jour` est le nombre de jours nécessaires.

**Résultat attendu** :
```
5
```

<details><summary>💡 Indice</summary>

Condition : `cumul < 50`. Avance : `jour = jour + 1`. Si la cellule ne s'arrête pas, clique sur ■ (stop) : c'est une boucle infinie.

📖 Module 04, cours (`while`).

</details>

In [ ]:
import pandas as pd

meteo = pd.read_csv("../donnees/meteo.csv")
pluies = list(meteo[meteo["ville"] == "Albi"]["pluie_mm"])
cumul = 0
jour = 0
while ...:
    cumul = cumul + pluies[jour]
    jour = ...
print(jour)

In [ ]:
# Vérification
assert "jour" in dir(), "Exécute d'abord ta cellule."
assert jour == 5, f"jour doit valoir 5, tu as {jour} : while cumul < 50, et jour = jour + 1 dans la boucle."
print("✅ OK")

## 🏁 Bravo !

Tu sais créer une base, y ranger des données en toute sécurité avec `?`, les interroger, les corriger et les
récupérer dans pandas. Coche la checklist du `README.md`, puis passe au fil rouge : `03-fil-rouge.ipynb`.